To mount google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# LIBRARIES INSTALLATION

In [ ]:
!pip install mapbox-vector-tile mercantile

# PHASE 1: TILE SCRAPER

In [ ]:
import requests
import mapbox_vector_tile
from shapely.geometry import shape
import csv
import math
import pandas as pd
import concurrent.futures
from tqdm.notebook import tqdm
import os

# -------------------------------
# CONFIGURATION
# -------------------------------
ZOOM = 15
EXTENT = 4096
OUTPUT_DIR = "/content/drive/MyDrive/suhail/"
MAX_WORKERS = 16

# -------------------------------
# REGION DATABASE
# Format: "Display Name": ("url_slug", (NORTH, WEST, SOUTH, EAST))
# -------------------------------
REGION_CONFIG = {
    "1": ("Riyadh", "riyadh", (25.15, 46.40, 24.40, 47.10)),
    "2": ("Eastern Region (Dammam/Khobar)", "eastern_region", (26.60, 49.90, 26.20, 50.25)),
    "3": ("Madinah", "al_madenieh", (24.58, 39.45, 24.35, 39.75)),
    "4": ("Makkah", "makkah_region", (21.55, 39.70, 21.25, 40.05)),
    "5": ("Al Qassim (Buraydah)", "al_qassim", (26.45, 43.85, 26.25, 44.05)),
    "6": ("Asir (Abha)", "asir_region", (18.35, 42.35, 18.15, 42.60)),
    "7": ("Tabuk", "tabuk", (28.48, 36.45, 28.32, 36.65)),
    "8": ("Hail", "hail", (27.60, 41.60, 27.40, 41.80)),
    "9": ("Najran", "najran", (17.58, 44.08, 17.45, 44.25)),
    "10": ("Jazan", "jazan", (16.95, 42.50, 16.82, 42.65)),
    "11": ("Al Jouf (Sakaka)", "jawf", (30.00, 40.10, 29.85, 40.25)),
    "12": ("Northern Borders (Arar)", "northern_borders", (30.99, 41.00, 30.90, 41.15)),
    "13": ("Al Bahah", "bahah", (20.06, 41.42, 19.98, 41.52))
}

# -------------------------------
# INPUT SELECTION
# -------------------------------
print("Available Regions:")
for key, val in REGION_CONFIG.items():
    print(f"{key}: {val[0]}")

selection = input("\nEnter the number of the region to scrape: ").strip()

if selection not in REGION_CONFIG:
    print("Invalid selection. Please run the cell again.")
else:
    region_name, slug, bounds = REGION_CONFIG[selection]
    north, west, south, east = bounds

    output_csv = f"{OUTPUT_DIR}suhail_{slug}_centroids.csv"
    base_url = f"https://tiles.suhail.ai/maps/{slug}/{{z}}/{{x}}/{{y}}.vector.pbf"

    print(f"\n--- SELECTED: {region_name} ---")
    print(f"Slug: {slug}")
    print(f"Bounds: {bounds}")
    print(f"Target File: {output_csv}")

    # -------------------------------
    # LOGIC
    # -------------------------------
    def lonlat_to_tile(lon, lat, z):
        n = 2 ** z
        x_tile = int((lon + 180.0) / 360.0 * n)
        lat_rad = math.radians(lat)
        y_tile = int((1 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2 * n)
        return x_tile, y_tile

    def tile_coords_to_lonlat(px, py, z, tile_x, tile_y, extent=EXTENT):
        fx = px / extent
        fy = (extent - py) / extent # Flip Y for MVT
        n = 2 ** z
        x_frac = tile_x + fx
        y_frac = tile_y + fy
        lon = (x_frac / n) * 360.0 - 180.0
        lat_rad = math.atan(math.sinh(math.pi * (1 - 2 * y_frac / n)))
        lat = math.degrees(lat_rad)
        return lon, lat

    def process_tile(tile_info):
        x, y = tile_info
        centroids = []
        tile_url = base_url.format(z=ZOOM, x=x, y=y)

        try:
            response = requests.get(tile_url, timeout=5)
            try:
                content = response.content
            except:
                return centroids

            if response.status_code != 200:
                return centroids

            tile = mapbox_vector_tile.decode(content)

            if tile:
                layer_name = next(iter(tile))
                for feature in tile[layer_name].get("features", []):
                    props = feature.get("properties", {})

                    oid = props.get("parcel_objectid") or props.get("OBJECTID")
                    pid = props.get("parcel_id") or props.get("id")

                    if not pid and oid:
                        pid = oid

                    if pid and oid and feature.get("geometry"):
                        try:
                            poly = shape(feature.get("geometry"))
                            c = poly.centroid
                            lon, lat = tile_coords_to_lonlat(c.x, c.y, ZOOM, x, y)
                            centroids.append({"parcel_id": pid, "parcel_objectid": oid, "lon": lon, "lat": lat})
                        except: pass
        except:
            pass
        return centroids

    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)

    x_min, y_min = lonlat_to_tile(west, north, ZOOM)
    x_max, y_max = lonlat_to_tile(east, south, ZOOM)
    x_start, x_end = min(x_min, x_max), max(x_min, x_max)
    y_start, y_end = min(y_min, y_max), max(y_min, y_max)

    tiles = [(x, y) for x in range(x_start, x_end + 1) for y in range(y_start, y_end + 1)]
    print(f"Scanning {len(tiles)} tiles...")

    all_data = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        results = list(tqdm(executor.map(process_tile, tiles), total=len(tiles)))

    for r in results: all_data.extend(r)

    if all_data:
        df = pd.DataFrame(all_data).drop_duplicates(subset=['parcel_id'])
        df.to_csv(output_csv, index=False)
        print(f"\n✅ DONE! Saved {len(df)} unique parcels to: {output_csv}")
        last_generated_file = output_csv # Pass to next cell
    else:
        print("\n❌ No parcels found. Check slug or coordinates.")

#TO DOWNLOAD BUILDING DATA FOR EVERY PARCELID THAT WE GOT FROM TILES

In [ ]:
import pandas as pd
import asyncio
import aiohttp
import os
import glob
from tqdm.asyncio import tqdm

# -------------------------------
# CONFIGURATION
# -------------------------------
try:
    INPUT_CSV = last_generated_file
    print(f"Auto-detected input file: {INPUT_CSV}")
except NameError:
    INPUT_CSV = input("Enter path to centroids CSV: ").strip()

OUTPUT_BASE = INPUT_CSV.replace(".csv", "_full_part")
API_URL = "https://api2.suhail.ai/parcel/buildingRules?parcelObjectId={}"
CONCURRENT_LIMIT = 40  # Number of parallel requests
MAX_ROWS_PER_FILE = 900000
SAVE_EVERY = 1000  # Save to disk every 1000 rows

# -------------------------------
# SETUP & RESUME LOGIC
# -------------------------------
FINAL_COLUMNS = [
    'parcel_id', 'parcel_objectid', 'lon', 'lat', 'rule_id',
    'zoningId', 'zoningColor', 'landuse', 'description',
    'maxBuildingCoefficient', 'maxBuildingHeight', 'maxParcelCoverage',
    'mainStreetsSetback', 'sideRearSetback', 'api_status', 'extra_data'
]

if not os.path.exists(INPUT_CSV):
    raise FileNotFoundError(f"File not found: {INPUT_CSV}")

print("Loading input list...")
df_input = pd.read_csv(INPUT_CSV, dtype=str)
if 'parcel_objectid' not in df_input.columns:
    print("Error: Input CSV must have 'parcel_objectid'")
    exit()

# --- INTELLIGENT RESUME LOGIC ---
# Find all output files (part1, part2, etc.) to see what we already finished
existing_files = glob.glob(f"{OUTPUT_BASE}*.csv")
processed_ids = set()

if existing_files:
    print(f"Found {len(existing_files)} existing output files. Scanning for completed IDs...")
    for fpath in existing_files:
        try:
            # We only need the ID column to know it's done
            done_df = pd.read_csv(fpath, usecols=['parcel_objectid'], dtype=str)
            processed_ids.update(done_df['parcel_objectid'].tolist())
        except: pass
    print(f"✅ Found {len(processed_ids)} already processed. Skipping them.")

# Filter input: Only take IDs that are NOT in processed_ids
df_todo = df_input[~df_input['parcel_objectid'].isin(processed_ids)]
print(f"Remaining parcels to fetch: {len(df_todo)}")

if len(df_todo) == 0:
    print("🎉 All data collected! Nothing to do.")
    exit()
# -------------------------------

class FileHandler:
    def __init__(self, base_path, limit):
        self.base_path = base_path
        self.limit = limit
        self.part_num = 1
        self.current_rows = 0
        self.current_file = f"{self.base_path}{self.part_num}.csv"
        self._find_latest_file()

    def _find_latest_file(self):
        # Find the last part file to append to
        while os.path.exists(f"{self.base_path}{self.part_num}.csv"):
             # Check if full
             try:
                 count = sum(1 for _ in open(f"{self.base_path}{self.part_num}.csv", encoding='utf-8-sig')) - 1
             except: count = 0

             if count >= self.limit:
                 self.part_num += 1
             else:
                 self.current_rows = count
                 break
        self.current_file = f"{self.base_path}{self.part_num}.csv"
        print(f"Resuming write at: {self.current_file} (Rows: {self.current_rows})")

    def save(self, df_chunk):
        if df_chunk.empty: return

        if self.current_rows + len(df_chunk) > self.limit:
            space_left = self.limit - self.current_rows
            if space_left > 0:
                self._write(df_chunk.iloc[:space_left])

            self.part_num += 1
            self.current_rows = 0
            self.current_file = f"{self.base_path}{self.part_num}.csv"
            print(f"--- Switching to new file: {self.current_file} ---")

            self.save(df_chunk.iloc[space_left:])
        else:
            self._write(df_chunk)

    def _write(self, df):
        is_new = not os.path.exists(self.current_file) or os.stat(self.current_file).st_size == 0
        df.to_csv(self.current_file, mode='a', header=is_new, index=False, encoding='utf-8-sig')
        self.current_rows += len(df)

file_handler = FileHandler(OUTPUT_BASE, MAX_ROWS_PER_FILE)

# -------------------------------
# ASYNC WORKER
# -------------------------------
async def fetch(session, row, sem):
    async with sem:
        pid = row.get('parcel_objectid')
        if not pid or str(pid).lower() == 'nan': return []

        base = {k: row.get(k) for k in ['parcel_id', 'parcel_objectid', 'lon', 'lat']}

        try:
            async with session.get(API_URL.format(pid), timeout=25) as resp:
                if resp.status == 200:
                    try:
                        data = await resp.json()
                        results = []
                        items = data.get('data', [])
                        if not items: return [dict(base, api_status="NO_RULES")]

                        for item in items:
                            new_row = base.copy()
                            for key in ['id', 'zoningId', 'zoningColor', 'landuse', 'description',
                                        'maxBuildingCoefficient', 'maxBuildingHeight', 'maxParcelCoverage']:
                                if key in item: new_row[key if key != 'id' else 'rule_id'] = item[key]

                            if 'mainStreetsSetback' in item: new_row['mainStreetsSetback'] = item['mainStreetsSetback']
                            if 'sideRearSetback' in item: new_row['sideRearSetback'] = item['sideRearSetback']

                            new_row['api_status'] = 'SUCCESS'
                            results.append(new_row)
                        return results
                    except: return [dict(base, api_status="JSON_ERROR")]
                elif resp.status == 429:
                    await asyncio.sleep(2)
                    return [dict(base, api_status="RATE_LIMIT")]
                elif resp.status == 404:
                    return [dict(base, api_status="NOT_FOUND")]
                else:
                    return [dict(base, api_status=f"HTTP_{resp.status}")]
        except:
            return [dict(base, api_status="CONN_ERROR")]

async def main():
    sem = asyncio.Semaphore(CONCURRENT_LIMIT)
    records = df_todo.to_dict('records')

    async with aiohttp.ClientSession() as session:
        tasks = [fetch(session, row, sem) for row in records]
        buffer = []

        for f in tqdm.as_completed(tasks, total=len(tasks), desc="Fetching"):
            res = await f
            buffer.extend(res)

            if len(buffer) >= SAVE_EVERY:
                df_chunk = pd.DataFrame(buffer).reindex(columns=FINAL_COLUMNS)
                file_handler.save(df_chunk)
                buffer = []

        if buffer:
            df_chunk = pd.DataFrame(buffer).reindex(columns=FINAL_COLUMNS)
            file_handler.save(df_chunk)

if __name__ == "__main__":
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None

    if loop and loop.is_running():
        await main()
    else:
        asyncio.run(main())

    print(f"\n✅ Job Complete!")